# S13 · Integración del sistema, DataOps y BI
**Escudo Andino · anticipación de heladas en Juliaca** · Big Data UPeU 2026-2 · Equipo 10 BIG IMPACT

Una sola ejecución integrada que: comprueba sus dependencias (compuertas previas), procesa el **batch** con compuertas de calidad, procesa el **streaming** con el modelo de S10 sobre Kafka, publica sus resultados en Prometheus/Grafana y deja un **manifiesto** de la ejecución.

Orden de uso: ejecutar las celdas de arriba hacia abajo. Si una compuerta falla, la ejecución se detiene con el nombre de lo que falta.

## 3.1 Reanudar el entorno completo
**Producto del paso:** los grupos de contenedores arriba y el productor publicando. Desde la raíz del repositorio, en PowerShell (solo si algo está caído):

```powershell
docker compose -f kafka/compose.yml up -d --wait
docker compose -f pyspark/compose.yml -f pyspark/compose.kafka.yml up -d
docker compose -f obs/compose.yml up -d --wait
docker compose -f uso-heladas/compose.yml up -d
docker exec -d lambda26-uso-heladas python producer_clima.py
```

Confirma que el productor publica leyendo el *offset* **dos veces** con unos minutos de diferencia (llega una lectura cada 15 min):

```powershell
docker exec lambda26-kafka /opt/kafka/bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic heladas-eventos
```

## 3.2 Crear la sesión de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("sesion13-integracion-dataops")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

## 3.3 Compuertas previas
**Producto del paso:** una celda que comprueba ocho dependencias y **detiene la ejecución** si alguna falla.

In [ ]:
import json, os, platform, time, urllib.request
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
INICIO = time.time()
manifiesto = {"runId": RUN_ID, "proyecto": "Escudo Andino - heladas en Juliaca",
              "sparkVersion": spark.version, "python": platform.python_version(),
              "compuertas": {}, "batch": {}, "streaming": {}, "observabilidad": {}}

RUTA_GOLD_U1 = "/opt/u1/s03-calidad-particionamiento/artifacts/clima_particionado"
RUTA_EVENTOS_S9 = "/opt/u2/s09-observabilidad-pipeline/artifacts/heladas_eventos_observabilidad"
RUTA_MODELO_S10 = "/opt/u2/s10-series-tiempo-inferencia-streaming/artifacts/modelo_rf_temp_h1"
TOPIC = "heladas-eventos"
MAX_EDAD_PRODUCTOR_MIN = 30      # el productor publica cada 15 min; 30 min = dos lecturas perdidas


def http_ok(url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False


def kafka_resumen(topic):
    df = (spark.read.format("kafka")
          .option("kafka.bootstrap.servers", "kafka:9092")
          .option("subscribe", topic)
          .option("startingOffsets", "earliest").option("endingOffsets", "latest")
          .load())
    r = df.agg(F.count("*").alias("filas"), F.max(F.unix_millis("timestamp")).alias("ultimoMs")).first()
    return r["filas"], r["ultimoMs"]


compuertas = {
    "Gold de U1 existe": os.path.isdir(RUTA_GOLD_U1),
    "Parquet de S9 existe": os.path.isdir(RUTA_EVENTOS_S9),
    "Modelo de S10 existe": os.path.isdir(RUTA_MODELO_S10),
    "Pushgateway listo": http_ok("http://pushgateway:9091/-/ready"),
    "Prometheus listo": http_ok("http://prometheus:9090/-/ready"),
    "Grafana saludable": http_ok("http://grafana:3000/api/health"),
}
try:
    filas_topic, ultimo_ms = kafka_resumen(TOPIC)
except Exception as e:
    filas_topic, ultimo_ms = 0, None
edad_min = None if ultimo_ms is None else (time.time() * 1000 - ultimo_ms) / 60000
compuertas["Kafka: topic " + TOPIC + " con datos"] = filas_topic > 0
compuertas["Productor activo (ultimo evento < %d min)" % MAX_EDAD_PRODUCTOR_MIN] = (
    edad_min is not None and edad_min < MAX_EDAD_PRODUCTOR_MIN)

for nombre, ok in compuertas.items():
    print(("OK    " if ok else "FALLA ") + nombre)
print("Eventos en el topic:", filas_topic, "| edad del ultimo evento (min):",
      None if edad_min is None else round(edad_min, 1))

manifiesto["compuertas"] = compuertas
manifiesto["kafkaFilasTopic"] = filas_topic
manifiesto["edadUltimoEventoMin"] = None if edad_min is None else round(edad_min, 1)
fallidas = [n for n, ok in compuertas.items() if not ok]
assert not fallidas, "Compuertas que fallan: " + str(fallidas)

Los nombres de host (`pushgateway`, `prometheus`, `grafana`, `kafka`) son los de la red Docker `lambda26-kafka-net`. **Error frecuente:** que "Gold de U1", "Parquet de S9" o "Modelo de S10" falle en una copia clonada: `artifacts/` está en el `.gitignore`, esos artefactos se generan ejecutando S3, S9 y S10.

## 3.4 Batch: KPI con compuertas de calidad y de plausibilidad
**Producto del paso:** (a) los KPI del Gold de U1 (146 832 horas reales) y (b) los KPI de latencia del Parquet de S9 con **percentiles**, no solo promedio.

In [ ]:
# (a) Gold de U1: calidad e hidrologia basica de la serie historica
COLS = ["temperatura", "puntoRocio", "humedad", "nubosidad"]
gold = (spark.read.parquet(RUTA_GOLD_U1)
        .filter(F.col("hora") <= F.lit("2026-10-01 23:59:59").cast("timestamp")))
n_gold = gold.count()
nulos_gold = gold.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in COLS]).first()
g = gold.agg(
    F.min("hora").alias("desde"), F.max("hora").alias("hasta"),
    F.min("temperatura").alias("tMin"), F.max("temperatura").alias("tMax"),
    F.avg("temperatura").alias("tMedia"),
    (F.avg((F.col("temperatura") <= 0).cast("int")) * 100).alias("pctHelada"),
    F.sum((F.col("temperatura") <= 0).cast("int")).alias("horasHelada"),
).first()

calidad_gold = {
    "Gold: hay filas (>= 100 000)": n_gold >= 100000,
    "Gold: sin nulos en las 4 variables": all((nulos_gold[c] or 0) == 0 for c in COLS),
    "Gold: temperatura en rango plausible (-30 a 40 C)": -30 <= g["tMin"] and g["tMax"] <= 40,
    "Gold: % de horas con helada entre 0 y 30 %": 0 <= g["pctHelada"] <= 30,
}
for nombre, ok in calidad_gold.items():
    print(("OK    " if ok else "FALLA ") + nombre)
assert all(calidad_gold.values()), "La calidad del Gold no pasa"
print(f"Gold: {n_gold:,} horas | {g['desde']} -> {g['hasta']}")
print(f"Temperatura: min {g['tMin']:.1f} | media {g['tMedia']:.2f} | max {g['tMax']:.1f} C")
print(f"Horas con helada: {g['horasHelada']:,} ({g['pctHelada']:.2f} %)")

In [ ]:
# (b) Parquet de S9: latencia de la ingesta. Percentiles: nunca publicar solo el promedio
eventos = spark.read.parquet(RUTA_EVENTOS_S9)
total = eventos.count()
nulos = eventos.select(
    F.sum(F.col("isValid").isNull().cast("int")).alias("isValid"),
    F.sum(F.col("latencyMs").isNull().cast("int")).alias("latencyMs"),
).first()
kpi = eventos.agg(
    F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
    F.avg("latencyMs").alias("latenciaPromedioMs"),
    F.expr("percentile_approx(latencyMs, 0.5)").alias("latenciaP50Ms"),
    F.expr("percentile_approx(latencyMs, 0.95)").alias("latenciaP95Ms"),
    F.max("latencyMs").alias("latenciaMaxMs"),
    F.avg((F.col("latencyMs") > 60000).cast("int")).alias("fraccionMayor1Min"),
).first()

compuertas_calidad = {
    "Hay filas": total > 0,
    "isValid sin nulos": (nulos["isValid"] or 0) == 0,
    "latencyMs sin nulos": (nulos["latencyMs"] or 0) == 0,
    "ratioValidos en [0, 1]": 0.0 <= kpi["ratioValidos"] <= 1.0,
    "Latencia tipica (p50) menor a 60 s": kpi["latenciaP50Ms"] < 60000,
}
for nombre, ok in compuertas_calidad.items():
    print(("OK    " if ok else "FALLA ") + nombre)
assert all(compuertas_calidad.values()), "La calidad del batch no pasa"

print(f"Eventos: {total:,}  |  ratio de validos: {kpi['ratioValidos']:.4f}")
print(f"Latencia: promedio {kpi['latenciaPromedioMs']:.0f} ms | p50 {kpi['latenciaP50Ms']} ms | "
      f"p95 {kpi['latenciaP95Ms']} ms | max {kpi['latenciaMaxMs']} ms")
print(f"Eventos con mas de 1 minuto de latencia: {kpi['fraccionMayor1Min']:.1%}")
advertencias = []
if kpi["latenciaP95Ms"] > 300000:
    advertencias.append("p95 de latencia mayor a 5 minutos: el promedio no representa a la mayoria de los eventos")
for a in advertencias:
    print("ADVERTENCIA:", a)

manifiesto["batch"] = {
    "gold": {"filas": n_gold, "desde": str(g["desde"]), "hasta": str(g["hasta"]),
             "tMin": g["tMin"], "tMedia": g["tMedia"], "tMax": g["tMax"],
             "pctHelada": g["pctHelada"], "horasHelada": g["horasHelada"], "calidad": calidad_gold},
    "eventosS9": {"filas": total, "ratioValidos": kpi["ratioValidos"],
                  "latenciaPromedioMs": kpi["latenciaPromedioMs"], "latenciaP50Ms": kpi["latenciaP50Ms"],
                  "latenciaP95Ms": kpi["latenciaP95Ms"], "latenciaMaxMs": kpi["latenciaMaxMs"],
                  "fraccionMayor1Min": kpi["fraccionMayor1Min"], "calidad": compuertas_calidad},
    "advertencias": advertencias,
}

**Por qué p50 y p95:** el promedio resume una distribución en un solo número y puede describir a nadie. La **compuerta** usa la mediana (qué vive un evento típico); el **p95** queda como *advertencia*, porque detener todo el pipeline por una cola larga sería una decisión de negocio, no técnica.

## 3.5 Streaming: el modelo de S10 sobre Kafka
**Producto del paso:** el modelo de S10 aplicado al topic `heladas-eventos` y las predicciones acumuladas.

Adaptación al proyecto: el productor publica **una lectura cada 15 min**, así que una ventana de 45 s (como la guía de ejemplo, que recibe un evento cada 3 s) no vería ningún evento. La ejecución integrada procesa **todo lo publicado en el topic** con `trigger(availableNow=True)`: el *stream* lee lo disponible, lo procesa y **termina**, que es lo que exige una ejecución comprobable. La vivacidad del productor ya quedó comprobada en 3.3.

In [ ]:
from pyspark.ml import PipelineModel
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, LongType

modelo = PipelineModel.load(RUTA_MODELO_S10)

esquema_evento = StructType([
    StructField("tipoEvento", StringType()), StructField("ciudad", StringType()),
    StructField("temperatura", DoubleType()), StructField("puntoRocio", DoubleType()),
    StructField("humedad", IntegerType()), StructField("nubosidad", IntegerType()),
    StructField("origen", StringType()), StructField("observadoEn", LongType()),
    StructField("timestamp", LongType()),
])


def preparar(df_kafka):
    # mismo codigo que S10: JSON de Kafka -> columnas que espera el modelo
    return (df_kafka
            .select(F.from_json(F.col("value").cast("string"), esquema_evento).alias("e")).select("e.*")
            .filter(F.col("tipoEvento") == "clima.lectura")
            .withColumn("observado_utc", (F.col("observadoEn") / 1000).cast("timestamp"))
            .withColumn("hora_local", F.from_utc_timestamp("observado_utc", "America/Lima"))
            .withColumn("hora_dia", F.hour("hora_local"))
            .withColumn("mes", F.month("hora_local")))


def solo_validos(ev):
    return (ev.dropna(subset=["temperatura", "puntoRocio", "humedad", "nubosidad", "observadoEn"])
              .filter(F.col("temperatura").between(-50, 50)))


crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", "kafka:9092").option("subscribe", TOPIC)
         .option("startingOffsets", "earliest").option("failOnDataLoss", "false").load())

eventos_stream = (solo_validos(preparar(crudo))
                  .withWatermark("observado_utc", "2 hours")
                  .dropDuplicates(["observado_utc"]))
predicciones = (modelo.transform(eventos_stream)
                .select("observado_utc", "temperatura", F.col("pred_rf").alias("prediccion")))

acumulado = []
microlotes = []


def procesar_microlote(df, id_lote):
    filas = df.collect()
    microlotes.append({"batchId": id_lote, "filas": len(filas)})
    acumulado.extend((f["observado_utc"].timestamp(), f["temperatura"], f["prediccion"]) for f in filas)


t0 = time.time()
consulta = predicciones.writeStream.foreachBatch(procesar_microlote).trigger(availableNow=True).start()
consulta.awaitTermination(600)
if consulta.isActive:
    consulta.stop()
duracion_stream = round(time.time() - t0, 1)

print("Micro-lotes con datos:", len([m for m in microlotes if m["filas"] > 0]))
print("Predicciones generadas:", len(acumulado), "| duracion del stream:", duracion_stream, "s")

**Error frecuente:** el `assert` de 3.6 falla con `El stream no entrego ninguna fila`. Casi siempre es que el topic está vacío o que `RUTA_MODELO_S10` apunta a un modelo que no se guardó en S10.

## 3.6 Error de predicción en producción
**Producto del paso:** el error absoluto medio (MAE) del modelo frente a la temperatura **real una hora después**, comparado con la **persistencia** (repetir la última lectura), sobre lo que acaba de procesar el *stream*.

In [ ]:
assert acumulado, "El stream no entrego ninguna fila: esta corriendo el productor / hay datos en el topic?"

real_por_ts = {int(ts): temp for ts, temp, _ in acumulado}      # temperatura real por instante observado
errores_modelo, errores_persist = [], []
alertas = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}
for ts, temp, pred in sorted(acumulado):
    real_1h = real_por_ts.get(int(ts) + 3600)                    # lectura real exactamente una hora despues
    if real_1h is None:
        continue
    errores_modelo.append(abs(pred - real_1h))
    errores_persist.append(abs(temp - real_1h))
    alerta, helada = pred <= 0, real_1h <= 0
    alertas["tp" if alerta and helada else "fp" if alerta else "fn" if helada else "tn"] += 1

pares = len(errores_modelo)
mae = sum(errores_modelo) / pares if pares else None
mae_persist = sum(errores_persist) / pares if pares else None
n_alertas = sum(1 for _, _, p in acumulado if p <= 0)

print("Pares comparables (prediccion vs real a 1 h):", pares)
print("MAE modelo      :", None if mae is None else round(mae, 3), "C")
print("MAE persistencia:", None if mae_persist is None else round(mae_persist, 3), "C")
print("Alertas de helada (pred <= 0 C):", n_alertas, "| matriz de alertas:", alertas)
manifiesto["streaming"] = {"modo": "availableNow sobre el topic completo", "duracionS": duracion_stream,
                           "filas": len(acumulado), "microlotes": microlotes, "paresComparables": pares,
                           "maeModelo": mae, "maePersistencia": mae_persist,
                           "alertas": n_alertas, "matrizAlertas": alertas}

El MAE **cambia en cada corrida** porque el topic crece con lecturas nuevas: un *pipeline* con fuente en vivo no es reproducible valor por valor. Lo reproducible es el **procedimiento** (mismos pasos, compuertas y dependencias) y lo auditable es **qué dio cada corrida**: para eso existe el manifiesto de 3.8. Una regresión sería que las compuertas dejen de pasar o que el *stream* procese cero filas, no que el MAE cambie.

## 3.7 Publicar en Pushgateway y confirmar en Prometheus
**Producto del paso:** las métricas de la ejecución en Prometheus, confirmadas **contra el valor recién publicado** (no contra "algún valor").

In [ ]:
b, e = manifiesto["batch"], manifiesto["streaming"]
metricas = {
    "lambda26_int_gold_horas_total": b["gold"]["filas"],
    "lambda26_int_gold_helada_pct": b["gold"]["pctHelada"],
    "lambda26_int_batch_eventos_total": b["eventosS9"]["filas"],
    "lambda26_int_batch_validos_ratio": b["eventosS9"]["ratioValidos"],
    "lambda26_int_batch_latencia_p50_ms": b["eventosS9"]["latenciaP50Ms"],
    "lambda26_int_batch_latencia_p95_ms": b["eventosS9"]["latenciaP95Ms"],
    "lambda26_int_stream_filas_total": e["filas"],
    "lambda26_int_stream_mae": e["maeModelo"] if e["maeModelo"] is not None else 0,
    "lambda26_int_stream_mae_persistencia": e["maePersistencia"] if e["maePersistencia"] is not None else 0,
    "lambda26_int_stream_alertas_total": e["alertas"],
    "lambda26_int_ultima_ejecucion_timestamp": int(time.time()),
}
cuerpo = "".join("# TYPE " + n + " gauge\n" + n + " " + str(v) + "\n" for n, v in metricas.items())
url = "http://pushgateway:9091/metrics/job/lambda26_integracion_heladas/instance/s13"
with urllib.request.urlopen(urllib.request.Request(url, data=cuerpo.encode(), method="PUT")) as r:
    print("Pushgateway respondio:", r.status)


def consultar(nombre):
    res = json.load(urllib.request.urlopen("http://prometheus:9090/api/v1/query?query=" + nombre))["data"]["result"]
    return res[0]["value"][1] if res else None


def coincide(obtenido, esperado):
    return obtenido is not None and abs(float(obtenido) - float(esperado)) <= 1e-6 * max(1.0, abs(float(esperado)))


confirmadas = {}
limite = time.time() + 90
while time.time() < limite:
    confirmadas = {n: consultar(n) for n in metricas}
    if all(coincide(confirmadas[n], v) for n, v in metricas.items()):
        break
    time.sleep(5)
for n, v in metricas.items():
    print(n + ":", confirmadas[n], "(publicado:", v, ")")
todas = all(coincide(confirmadas[n], v) for n, v in metricas.items())
manifiesto["observabilidad"] = {"metricas": confirmadas, "todasEnPrometheus": todas}
assert todas, "Prometheus no devolvio los valores recien publicados"

El bucle espera hasta 90 s porque Prometheus recoge el dato en su siguiente *scrape*, no en el instante del `PUT`.

## 3.8 Escribir el manifiesto
**Producto del paso:** `artifacts/manifiesto_<runId>.json` con el registro de la ejecución (compuertas, KPI, MAE, duración, versiones).

In [ ]:
manifiesto["duracionTotalS"] = round(time.time() - INICIO, 1)
os.makedirs("./artifacts", exist_ok=True)
ruta_manifiesto = "./artifacts/manifiesto_" + RUN_ID + ".json"
with open(ruta_manifiesto, "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, indent=2, ensure_ascii=False, default=str)
print("Manifiesto escrito en", ruta_manifiesto)
print(json.dumps({k: manifiesto[k] for k in ("runId", "sparkVersion", "duracionTotalS")}, indent=2))

`artifacts/` está en el `.gitignore`: el manifiesto es evidencia de **tu** corrida y lo guardas para la sustentación. Ábrelo y confirma que están todas las compuertas, los percentiles, el MAE y la duración.

## 3.9 Tablero de integración en Grafana
**Producto del paso:** el tablero *S13 - Integración Heladas Juliaca* con cuatro paneles de DataOps más dos de contexto.

La celda lo crea por la API de Grafana (usa `GRAFANA_ADMIN_USER` / `GRAFANA_ADMIN_PASSWORD` o `admin`/`admin`). Si prefieres armarlo a mano, estas son las consultas:

| Panel | Consulta PromQL |
|---|---|
| Latencia p50 | `lambda26_int_batch_latencia_p50_ms` |
| Latencia p95 | `lambda26_int_batch_latencia_p95_ms` |
| Ratio de eventos válidos | `lambda26_int_batch_validos_ratio` |
| Error del modelo en producción (MAE) | `lambda26_int_stream_mae` |
| MAE de persistencia | `lambda26_int_stream_mae_persistencia` |
| Antigüedad de la última ejecución | `time() - lambda26_int_ultima_ejecucion_timestamp` |

In [ ]:
import base64

GF = "http://grafana:3000"
usuario = os.environ.get("GRAFANA_ADMIN_USER", "admin")
clave = os.environ.get("GRAFANA_ADMIN_PASSWORD", "admin")
cabecera = {"Authorization": "Basic " + base64.b64encode((usuario + ":" + clave).encode()).decode(),
            "Content-Type": "application/json"}


def gf(ruta, datos=None):
    req = urllib.request.Request(GF + ruta, headers=cabecera,
                                 data=None if datos is None else json.dumps(datos).encode(),
                                 method="GET" if datos is None else "POST")
    return json.load(urllib.request.urlopen(req))


def panel(pid, titulo, tipo, expr, x, y, w=6, h=5, unidad=None, minimo=None, maximo=None, extra=None):
    campos = {"defaults": {"unit": unidad} if unidad else {"unit": "none"}, "overrides": []}
    if minimo is not None:
        campos["defaults"]["min"] = minimo
    if maximo is not None:
        campos["defaults"]["max"] = maximo
    objetivos = [{"expr": e, "legendFormat": l, "refId": chr(65 + i)} for i, (e, l) in enumerate(expr)]
    return {"id": pid, "title": titulo, "type": tipo, "gridPos": {"x": x, "y": y, "w": w, "h": h},
            "datasource": DS, "targets": objetivos, "fieldConfig": campos,
            "options": {"reduceOptions": {"calcs": ["lastNotNull"]}, **(extra or {})}}


try:
    fuentes = gf("/api/datasources")
    prom = next(d for d in fuentes if d["type"] == "prometheus")
    DS = {"type": "prometheus", "uid": prom["uid"]}
    paneles = [
        panel(1, "Latencia: p50 frente a p95 (ms)", "stat",
              [("lambda26_int_batch_latencia_p50_ms", "p50"), ("lambda26_int_batch_latencia_p95_ms", "p95")], 0, 0, unidad="ms"),
        panel(2, "Ratio de eventos validos", "gauge", [("lambda26_int_batch_validos_ratio", "validos")], 6, 0,
              minimo=0, maximo=1, unidad="percentunit"),
        panel(3, "Error del modelo en produccion: MAE (C)", "stat",
              [("lambda26_int_stream_mae", "modelo"), ("lambda26_int_stream_mae_persistencia", "persistencia")], 12, 0),
        panel(4, "Antiguedad de la ultima ejecucion", "stat",
              [("time() - lambda26_int_ultima_ejecucion_timestamp", "hace")], 18, 0, unidad="s"),
        panel(5, "Horas reales en el Gold (U1)", "stat", [("lambda26_int_gold_horas_total", "horas")], 0, 5),
        panel(6, "% de horas con helada (2010-2026)", "stat", [("lambda26_int_gold_helada_pct", "helada")], 6, 5, unidad="percent"),
        panel(7, "Predicciones procesadas por el stream", "stat", [("lambda26_int_stream_filas_total", "filas")], 12, 5),
        panel(8, "Alertas de helada generadas", "stat", [("lambda26_int_stream_alertas_total", "alertas")], 18, 5),
    ]
    tablero = {"dashboard": {"uid": "lambda26-integracion-heladas", "title": "S13 - Integracion Heladas Juliaca",
                             "tags": ["lambda26", "heladas", "s13"], "timezone": "browser", "schemaVersion": 39,
                             "refresh": "30s", "time": {"from": "now-6h", "to": "now"}, "panels": paneles},
               "overwrite": True, "message": "S13 " + RUN_ID}
    res = gf("/api/dashboards/db", tablero)
    print("Tablero creado:", res.get("status"), "-> http://localhost:43000" + res.get("url", ""))
    manifiesto["observabilidad"]["tablero"] = res.get("url")
except Exception as err:
    print("No se pudo crear el tablero por API (", err, "). Armalo a mano con la tabla de arriba.")

In [ ]:
# vuelve a escribir el manifiesto con el dato del tablero
with open(ruta_manifiesto, "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, indent=2, ensure_ascii=False, default=str)
print("Manifiesto actualizado:", ruta_manifiesto)

## 3.10 Registrar los hallazgos en la bitácora
**Producto del paso:** una fila por hallazgo, con su causa, corrección y la prueba de la corrección. Las dos primeras se generan solas desde el manifiesto; agrega las tuyas.

In [ ]:
from IPython.display import Markdown, display

filas = []
lat = manifiesto["batch"]["eventosS9"]
prom_ms, p50, p95 = lat["latenciaPromedioMs"], lat["latenciaP50Ms"], lat["latenciaP95Ms"]
if manifiesto["batch"]["advertencias"]:
    filas.append(("Latencia con cola larga: el promedio no representa al evento tipico.",
                  "Se media solo el promedio; las compuertas de nulos y rango no lo detectan.",
                  "Percentiles p50/p95, fraccion > 1 min, compuerta sobre p50 y advertencia sobre p95 (3.4).",
                  f"La celda de 3.4 imprime promedio {prom_ms:.0f} ms, p50 {p50} ms y p95 {p95} ms."))
else:
    filas.append(("El KPI de latencia podia esconder una cola larga detras de un promedio aceptable.",
                  "Con un solo promedio no se ve la distribucion.",
                  "Se publican p50 y p95 y la fraccion > 1 min (3.4) en vez del promedio.",
                  f"p50 {p50} ms y p95 {p95} ms, ambos medidos y publicados en Prometheus."))
filas.append(("Una copia clonada del repositorio no puede ejecutar la integracion: faltan Gold, Parquet de S9 y modelo de S10.",
              "`artifacts/` esta en el `.gitignore`: no viaja con `git clone`.",
              "Compuertas previas que nombran el artefacto que falta (3.3).",
              "Con el artefacto ausente, la celda de 3.3 se detiene con `Compuertas que fallan: [...]`."))
filas.append(("El productor publica cada 15 min: una ventana fija de 45 s no recibe eventos.",
              "La guia de ejemplo asume un evento cada 3 s; WeatherAPI actualiza cada ~15 min.",
              "El stream usa `availableNow` sobre el topic y la vivacidad se comprueba con la edad del ultimo evento (3.3, 3.5).",
              f"Streaming procesado: {manifiesto['streaming']['filas']} predicciones y {manifiesto['streaming']['paresComparables']} pares con MAE."))

tabla = "| # | Hallazgo | Causa | Correccion | Prueba de correccion |\n|---|---|---|---|---|\n"
for i, f in enumerate(filas, 1):
    tabla += f"| {i} | " + " | ".join(f) + " |\n"
display(Markdown(tabla))
manifiesto["bitacora"] = [dict(zip(("hallazgo", "causa", "correccion", "prueba"), f)) for f in filas]
with open(ruta_manifiesto, "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, indent=2, ensure_ascii=False, default=str)
print("Bitacora guardada en el manifiesto:", ruta_manifiesto)

### Evidencia de aprendizaje
- Entorno levantado y productor publicando (3.1) · compuertas previas con sus resultados (3.3).
- KPI del Gold y de la latencia con percentiles, y compuertas de calidad (3.4).
- Streaming con el modelo de S10 y MAE de producción frente a persistencia (3.5-3.6).
- Métricas confirmadas en Prometheus (3.7) · manifiesto (3.8) · tablero de integración (3.9) · bitácora (3.10).

**Siguiente:** S14 (revisión técnica y hardening) repite esta ejecución como **prueba de no regresión**.